In [41]:
import pandas as pd
import numpy as np
import os

In [42]:
telco_df = pd.read_csv('../data/processed/telco_clean.csv')

print(telco_df.head())
print(telco_df.shape)

   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   
3       1              0        0           0      45             0   
4       0              0        0           0       2             1   

   MultipleLines  OnlineSecurity  OnlineBackup  DeviceProtection  ...  \
0              0               0             1                 0  ...   
1              0               1             0                 1  ...   
2              0               1             1                 0  ...   
3              0               1             0                 1  ...   
4              0               0             0                 0  ...   

   InternetService_Fiber optic  InternetService_No  Contract_Month-to-month  \
0                            0                   0     

### Feature considered and rejected: TotalServices

Summing the 7 service columns (MultipleLines, OnlineSecurity, OnlineBackup, DeviceProtection,
TechSupport, StreamingTV, StreamingMovies) into a single count produced a correlation with
Churn of only -0.070 — weaker than several of the individual columns it was built from
(OnlineSecurity alone: -0.171, TechSupport alone: -0.165). Services aren't equally predictive;
summing them dilutes the strong signals with the weak ones. Dropped in favor of keeping the
7 columns separate, letting the model weigh each one individually.

In [35]:
telco_df['AvgMonthlySpend'] = telco_df['TotalCharges'] / telco_df['tenure']

print(telco_df['AvgMonthlySpend'].isnull().sum())
print(telco_df['AvgMonthlySpend'].describe())
print()
print(telco_df[['AvgMonthlySpend', 'MonthlyCharges']].head(10))

0
count    7032.000000
mean       64.799424
std        30.185891
min        13.775000
25%        36.179891
50%        70.373239
75%        90.179560
max       121.400000
Name: AvgMonthlySpend, dtype: float64

   AvgMonthlySpend  MonthlyCharges
0        29.850000           29.85
1        55.573529           56.95
2        54.075000           53.85
3        40.905556           42.30
4        75.825000           70.70
5       102.562500           99.65
6        88.609091           89.10
7        30.190000           29.75
8       108.787500          104.80
9        56.257258           56.15


In [36]:
cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
telco_df['HasProtectionBundle'] = np.where(telco_df[cols].sum(axis=1) >= 2, 1, 0)

print(telco_df['HasProtectionBundle'].value_counts())
print(telco_df.groupby('HasProtectionBundle')['Churn'].mean())


HasProtectionBundle
0    4254
1    2778
Name: count, dtype: int64
HasProtectionBundle
0    0.329337
1    0.168467
Name: Churn, dtype: float64


In [37]:
telco_df['IsNewCustomer'] = np.where(telco_df['tenure'] <= 12, 1, 0)

print(telco_df['IsNewCustomer'].value_counts())
print(telco_df.groupby('IsNewCustomer')['Churn'].mean())

IsNewCustomer
0    4857
1    2175
Name: count, dtype: int64
IsNewCustomer
0    0.171299
1    0.476782
Name: Churn, dtype: float64


In [38]:
corr = telco_df.corr()['Churn'].sort_values(ascending=False)

print("Top 10 correlations:")
print(corr.head(10))

print("\nBottom 10 correlations:")
print(corr.tail(10))

print("\nKey engineered features:")
print(corr[['AvgMonthlySpend', 'HasProtectionBundle', 'IsNewCustomer']])

Top 10 correlations:
Churn                             1.000000
Contract_Month-to-month           0.404565
IsNewCustomer                     0.319628
InternetService_Fiber optic       0.307463
PaymentMethod_Electronic check    0.301455
MonthlyCharges                    0.192858
AvgMonthlySpend                   0.192033
PaperlessBilling                  0.191454
SeniorCitizen                     0.150541
StreamingTV                       0.063254
Name: Churn, dtype: float64

Bottom 10 correlations:
Partner               -0.149982
Dependents            -0.163128
TechSupport           -0.164716
OnlineSecurity        -0.171270
HasProtectionBundle   -0.178027
Contract_One year     -0.178225
TotalCharges          -0.199484
InternetService_No    -0.227578
Contract_Two year     -0.301552
tenure                -0.354049
Name: Churn, dtype: float64

Key engineered features:
AvgMonthlySpend        0.192033
HasProtectionBundle   -0.178027
IsNewCustomer          0.319628
Name: Churn, dtype: float6

In [40]:
print(telco_df.isnull().sum().sum())
print(telco_df.shape)
print(telco_df[['AvgMonthlySpend', 'HasProtectionBundle', 'IsNewCustomer']].dtypes)

0
(7032, 30)
AvgMonthlySpend        float64
HasProtectionBundle      int64
IsNewCustomer            int64
dtype: object


In [39]:
os.makedirs('../data/processed', exist_ok=True)
telco_df.to_csv('../data/processed/telco_features.csv', index=False)

print('Saved to ../data/processed/telco_features.csv')
print(f'File size: {os.path.getsize('../data/processed/telco_features.csv')}')

Saved to ../data/processed/telco_features.csv
File size: 585557


## Feature Engineering Summary

| Feature | Type | Description | Correlation with Churn |
|---------|------|--------------|------------------------|
| AvgMonthlySpend | float | TotalCharges / tenure — average historical monthly spend, distinct from current MonthlyCharges | 0.192 |
| HasProtectionBundle | binary | 1 if customer has 2+ of OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport | -0.178 |
| IsNewCustomer | binary | 1 if tenure ≤ 12 months | 0.320 |

**Rejected:** TotalServices (sum of 7 service columns) — correlation (-0.070) was weaker than
several of its constituent columns, indicating the aggregation diluted signal rather than
concentrating it.

**Strongest finding:** IsNewCustomer ranks 3rd highest correlation in the entire dataset
(behind Contract_Month-to-month and ahead of InternetService_Fiber optic), validating the
tenure-risk pattern identified in EDA.

**Final dataset:** 7,032 rows · 30 columns · 0 missing values · all numeric.
Saved to data/processed/telco_features.csv.

### Next step
Baseline modelling (05_baseline_model.ipynb) — train/test split, baseline classifier,
establish ROC-AUC benchmark before any tuning.